In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd
import torch
import pydicom
import matplotlib.pyplot as plt

from pathlib import Path

# MONAI imports
import monai
from monai.data import Dataset, CacheDataset, DataLoader, PILReader
from monai.transforms import (
    LoadImage, LoadImaged, Resized, Compose, SaveImage, 
    Spacingd, SpatialCropd, ResizeWithPadOrCropd
)

import numpy as np
from monai.transforms import (
    Compose,
    LoadImaged,
    Transposed,
    NormalizeIntensityd,
    MapTransform,
#    ScaleIntensityRangePercentiled,
    ScaleIntensityRangePercentilesd
)


import landmarker
import landmarker.datasets
from landmarker.datasets import get_cepha_landmark_datasets

from landmarker.heatmap import GaussianHeatmapGenerator

import torch
from landmarker.models import OriginalSpatialConfigurationNet
from landmarker.losses import GaussianHeatmapL2Loss
from torch.utils.data import DataLoader


In [ ]:
# from landmarker.data import LandmarkDataset

# # Initialize dataset
# dataset = LandmarkDataset(
#     imgs=image_paths,          # List of paths to your images
#     landmarks=landmarks_array, # NumPy array of shape (N, C, D)
#                              # N = number of samples
#                              # C = number of landmark classes
#                              # D = spatial dimensions (2 or 3)
#     spatial_dims=2,          # 2 for 2D images, 3 for 3D
#     transform=transforms,    # MONAI transforms for preprocessing
#     dim_img=(512, 512),     # Target image dimensions
#     class_names=names       # List of landmark class names
# )

In [ ]:


# Load the ISBI2015 cephalometric dataset
data_dir = "/home/cwatzenboeck/data/public"
train_ds, test1_ds, test2_ds = get_cepha_landmark_datasets(data_dir)

In [ ]:


heatmap_generator = GaussianHeatmapGenerator(
    nb_landmarks=19,        # Number of landmarks
    sigmas=3,              # Standard deviation for Gaussian distribution
    learnable=True,        # Enable adaptive heatmap parameters
    heatmap_size=(512, 512) # Output heatmap dimensions
)


In [ ]:


# Initialize model
model = OriginalSpatialConfigurationNet(
    in_channels=3,    # Number of input channels
    out_channels=19   # Number of landmarks
)

# Set up optimizer
optimizer = torch.optim.SGD([
    {'params': model.parameters(), "weight_decay": 1e-3},
    {'params': heatmap_generator.sigmas},
    {'params': heatmap_generator.rotation}
], lr=1e-6, momentum=0.99, nesterov=True)

# Define loss function
criterion = GaussianHeatmapL2Loss(alpha=5)

# Create data loader
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

# Training loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

for epoch in range(100):
    model.train()
    for batch in train_loader:
        images = batch["image"].to(device)
        landmarks = batch["landmark"].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        heatmaps = heatmap_generator(landmarks)
        loss = criterion(outputs, heatmap_generator.sigmas, heatmaps)

        loss.backward()
        optimizer.step()

In [ ]:
idl = iter(train_loader)


In [ ]:
X = next(idl)
X["image"].shape

In [ ]:
X = next(idl)
X["image"].shape

In [ ]:
X = next(idl)
X["image"].shape

In [ ]:
X["landmark"].shape